In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [2]:
decision_tree = DecisionTreeClassifier()
random_forest = RandomForestClassifier()

params_tree = [{'criterion' : ['gini', 'entropy'], 'max_depth' : [2, 5, 10, 15], 'min_samples_split' : [2, 5, 10, 15],
                'min_samples_leaf' : [2, 5, 10, 15]}]
params_forest = [{'n_estimators' : [100, 200], 'max_depth' : [None, 5, 10, 15, 20], 'min_samples_split' : [2, 5, 10],
                  'min_samples_leaf' : [1, 2, 5, 10], 'bootstrap' : [True, False]}]

search_tree = RandomizedSearchCV(decision_tree, params_tree, cv = 5, n_iter = 100)
search_forest = RandomizedSearchCV(random_forest, params_forest, cv = 5, n_iter = 30)

## Dataset 1: Instacart Dataset

In [4]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

data1_transformed.head()

,user_id,product_id,ordered,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
0,71,45,0.0,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050
1,71,117,1.0,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000
2,71,2078,0.0,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522
3,71,2825,0.0,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372
4,71,3376,1.0,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438


In [5]:
X = data1_transformed.drop(columns = ['ordered', 'user_id', 'product_id'])
y = data1_transformed['ordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

search_tree.fit(X_train, y_train)
print(f'Best Hyperparameters for Decision Tree: {search_tree.best_params_}')

search_forest.fit(X_train, y_train)
print(f'Best Hyperparameters for Random Forest: {search_forest.best_params_}')

Best Hyperparameters for Decision Tree: {'min_samples_split': 10, 'min_samples_leaf': 15, 'max_depth': 5, 'criterion': 'gini'}
Best Hyperparameters for Random Forest: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_depth': 10, 'bootstrap': True}


In [9]:
best_tree = DecisionTreeClassifier(min_samples_split = 10, min_samples_leaf = 15, max_depth = 5, criterion = 'gini')

best_tree.fit(X_train, y_train)

y_preds = best_tree.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'].rank(method = 'first'), q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.007,0.008,0.119
2,0.007,0.007,0.115
3,0.016,0.017,0.269
4,0.021,0.024,0.375
5,0.029,0.033,0.517
6,0.036,0.040,0.626
7,0.049,0.049,0.771
8,0.077,0.081,1.266
9,0.117,0.120,1.886


In [11]:
best_forest = RandomForestClassifier(n_estimators = 100, min_samples_split = 10, min_samples_leaf = 10, 
max_depth = 10, bootstrap = True)

best_forest.fit(X_train, y_train)

y_preds = best_forest.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.005,0.005,0.078
2,0.009,0.009,0.144
3,0.014,0.012,0.189
4,0.019,0.021,0.328
5,0.027,0.028,0.444
6,0.038,0.043,0.671
7,0.052,0.056,0.888
8,0.075,0.079,1.246
9,0.116,0.117,1.834


## Dataset 2: Amazon Reviews Dataset

In [3]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked', 'rating'])
y = data2_transformed['liked']

X.head()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,False,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,True,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,True,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,True,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,True,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

search_tree.fit(X_train, y_train)
print(f'Best Hyperparameters for Decision Tree: {search_tree.best_params_}')

search_forest.fit(X_train, y_train)
print(f'Best Hyperparameters for Random Forest: {search_forest.best_params_}')

Best Hyperparameters for Decision Tree: {'min_samples_split': 15, 'min_samples_leaf': 15, 'max_depth': 5, 'criterion': 'entropy'}
Best Hyperparameters for Random Forest: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 10, 'bootstrap': True}


In [5]:
best_tree = DecisionTreeClassifier(min_samples_split = 15, min_samples_leaf = 15, max_depth = 5, criterion = 'entropy')

best_tree.fit(X_train, y_train)

y_preds = best_tree.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'].rank(method = 'first'), q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_liked = ('Actual', 'mean'))

overall_liked = y_test.mean()

lift_table['lift'] = lift_table['avg_liked']/overall_liked

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_liked'] = lift_table['avg_liked'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_liked,lift
decile,,,
1,0.657,0.800,0.957
2,0.722,0.667,0.798
3,0.866,0.786,0.940
4,0.884,0.600,0.718
5,0.884,1.000,1.197
6,0.928,0.733,0.878
7,0.957,0.929,1.111
8,1.000,0.933,1.117
9,1.000,1.000,1.197


In [6]:
best_forest = RandomForestClassifier(n_estimators = 100, min_samples_split = 10, min_samples_leaf = 1, 
max_depth = 10, bootstrap = True)

best_forest.fit(X_train, y_train)

y_preds = best_forest.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_liked = ('Actual', 'mean'))

overall_liked = y_test.mean()

lift_table['lift'] = lift_table['avg_liked']/overall_liked

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_liked'] = lift_table['avg_liked'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_liked,lift
decile,,,
1,0.694,0.667,0.798
2,0.786,0.800,0.957
3,0.843,0.786,0.940
4,0.862,0.733,0.878
5,0.881,0.800,0.957
6,0.892,0.929,1.111
7,0.907,0.857,1.026
8,0.927,0.933,1.117
9,0.947,1.000,1.197


## Dataset 3: Ecommerce Dataset

In [18]:
data3 = pd.read_csv('/workspaces/DX799---HW-Assignments/data.csv', encoding = 'windows-1252')
data3['Purchased'] = 1

customer_id = data3['Customer ID'].values
segments = data3['Segment'].values
cities = data3['City'].values
state = data3['State'].values
region = data3['Region'].values
product_id = data3['Product ID'].values
categories = data3['Category'].values
subcategories = data3['Sub-Category'].values

def generate_row(df):
    row = {}

    row['Customer ID'] = np.random.choice(customer_id)
    cust_id = row['Customer ID']
    purchased = set(df[df['Customer ID'] == cust_id]['Product ID'])
    candidates = list(set(product_id) - purchased)

    row['Segment'] = np.random.choice(segments)
    row['State'] = np.random.choice(state)
    row['City'] = np.random.choice(cities)
    row['Region'] = np.random.choice(region)
    row['Product ID'] = np.random.choice(candidates)
    row['Category'] = np.random.choice(categories)
    row['Sub-Category'] = np.random.choice(subcategories)

    row['Sales'] = 0
    row['Discount'] = 0
    row['Quantity'] = 0
    row['Profit'] = 0
    row['Purchased'] = 0

    return row

fake_rows = pd.DataFrame([generate_row(data3) for n in range(1000)])

fake_df = pd.concat([data3, fake_rows], ignore_index = True)

fake_df = fake_df.drop(columns = ['Row ID', 'Product Name', 'Country', 'Postal Code', 'Order ID', 'Order Date'])

fake_df_encoded = pd.get_dummies(fake_df, columns = ['Ship Mode', 'Segment', 'State', 'Region', 'Category', 'Sub-Category'], dtype = int)

X = fake_df_encoded.drop(columns = ['Customer ID', 'Product ID', 'City', 'Purchased'])
y = fake_df_encoded['Purchased']

X.head()

,Sales,Quantity,Discount,Profit,Ship Mode_First Class,Ship Mode_Same Day,Ship Mode_Second Class,Ship Mode_Standard Class,Segment_Consumer,Segment_Corporate,...,Sub-Category_Envelopes,Sub-Category_Fasteners,Sub-Category_Furnishings,Sub-Category_Labels,Sub-Category_Machines,Sub-Category_Paper,Sub-Category_Phones,Sub-Category_Storage,Sub-Category_Supplies,Sub-Category_Tables
0,48.896,4,0.2,8.5568,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
1,474.430,11,0.0,199.2606,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
2,3.600,2,0.0,1.7280,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,454.560,5,0.2,-107.9580,0,0,0,1,1,0,...,0,0,0,0,0,0,0,1,0,0
4,141.420,5,0.6,-187.3815,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

search_tree.fit(X_train, y_train)
print(f'Best Hyperparameters for Decision Tree: {search_tree.best_params_}')

search_forest.fit(X_train, y_train)
print(f'Best Hyperparameters for Random Forest: {search_forest.best_params_}')

Best Hyperparameters for Decision Tree: {'min_samples_split': 10, 'min_samples_leaf': 15, 'max_depth': 5, 'criterion': 'entropy'}
Best Hyperparameters for Random Forest: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_depth': 20, 'bootstrap': True}


In [20]:
best_tree = DecisionTreeClassifier(min_samples_split = 10, min_samples_leaf = 15, max_depth = 5, criterion = 'entropy')

best_tree.fit(X_train, y_train)

y_preds = best_tree.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'].rank(method = 'first'), q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_liked = ('Actual', 'mean'))

overall_liked = y_test.mean()

lift_table['lift'] = lift_table['avg_liked']/overall_liked

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_liked'] = lift_table['avg_liked'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_liked,lift
decile,,,
1,0.000,0.000,0.000
2,0.000,0.000,0.000
3,0.843,0.843,1.075
4,1.000,1.000,1.276
5,1.000,1.000,1.276
6,1.000,1.000,1.276
7,1.000,1.000,1.276
8,1.000,1.000,1.276
9,1.000,1.000,1.276


In [22]:
best_forest = RandomForestClassifier(n_estimators = 200, min_samples_split = 2, min_samples_leaf = 5, 
max_depth = 20, bootstrap = True)

best_forest.fit(X_train, y_train)

y_preds = best_forest.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'].rank(method = 'first'), q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_liked = ('Actual', 'mean'))

overall_liked = y_test.mean()

lift_table['lift'] = lift_table['avg_liked']/overall_liked

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_liked'] = lift_table['avg_liked'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_liked,lift
decile,,,
1,0.005,0.000,0.000
2,0.013,0.000,0.000
3,0.834,0.843,1.075
4,0.994,1.000,1.276
5,0.996,1.000,1.276
6,0.998,1.000,1.276
7,0.999,1.000,1.276
8,0.999,1.000,1.276
9,1.000,1.000,1.276


## Conclusion

Looking at the results, the results are very similar to the results I have gotten from the other models I have tested. The random forest and decision tree perform the best for the first dataset compared to other models. For dataset 2, it is fluctuating between deciles and there isn't a steady increase which was seen in other models tested. For dataset 3, the lift stayed the same after decile 3. I am hypothesizing the results for dataset 2 and 3 are due to the small population size and large class imbalance for these datasets.